# 05.0 SSL downstream models

## Notebook overview
This notebook rebuilds the saved SimCLR encoders from notebook 04, runs the downstream SSL anomaly detectors under matched settings, and combines those results with the baseline comparison tables created in notebooks 02 and 03.

## Purpose
- load the fixed split manifest and clean leakage checks created in notebook 01
- load the merged baseline table created in notebook 03 so the final comparison can be combined without rerunning the baseline notebooks
- rebuild the saved SimCLR encoders from notebook 04 and evaluate the downstream PatchCore and PaDiM variants
- save clean SSL-only and combined main comparison tables for later report writing and GitHub presentation
- export the main comparison figure, a category heatmap, and an efficiency summary table

## Process
1. resolve the dataset path and verify that notebooks 01, 03, and 04 were run successfully.
2. rebuild the shared dataloaders and evaluation helpers used for the downstream SSL stage.
3. reload the saved SimCLR encoder checkpoints and run the downstream PatchCore and PaDiM methods category by category.
4. merge the SSL rows with the baseline rows, save the final tables, and export the main comparison figures.


In [ ]:
#------------------------------------------------------------------------------
# 1.0 Optional dependency install
#------------------------------------------------------------------------------
try:
    import faiss
    print("faiss already available")
except Exception:
    !pip -q install -U faiss-cpu


In [ ]:
#------------------------------------------------------------------------------
# 1.1 Imports and notebook helpers
#------------------------------------------------------------------------------
import os
import sys
import json
import time
import math
import random
import hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms

from sklearn.metrics import roc_auc_score, average_precision_score

import faiss
from tqdm.auto import tqdm
from IPython.display import display


# Print a clean banner so long notebook output is easier to scan.
def print_banner(text):
    print("\n" + "=" * 90)
    print(text)
    print("=" * 90)


# Create the parent folder before saving an artefact.
def ensure_parent(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


# Save JSON with overwrite behaviour.
def save_json_overwrite(obj, path):
    path = ensure_parent(path)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


# Save CSV with overwrite behaviour.
def save_csv_overwrite(df, path):
    path = ensure_parent(path)
    df.to_csv(path, index=False)


# Read a JSON file from disk.
def read_json(path):
    with open(Path(path), "r") as f:
        return json.load(f)


# Return file size in MB if the path exists.
def file_size_mb(path):
    path = Path(path)
    return path.stat().st_size / (1024 ** 2) if path.exists() else np.nan


# Keep deterministic behaviour consistent with the earlier notebooks.
def set_seed(seed, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


# Build a stable category-specific seed from text.
def stable_seed_from_text(seed, text):
    key = f"{seed}_{text}".encode("utf-8")
    return int(hashlib.md5(key).hexdigest()[:8], 16)


# Reset tracked peak CUDA memory before one fit stage.
def reset_peak_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


# Read the tracked peak CUDA memory in MB.
def get_peak_gpu_memory_mb():
    return float(torch.cuda.max_memory_allocated() / (1024 ** 2)) if torch.cuda.is_available() else np.nan


print_banner("Environment")
print("Python        :", sys.version.split()[0])
print("Torch         :", torch.__version__)
print("Torchvision   :", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count     :", torch.cuda.device_count())
for gpu_idx in range(torch.cuda.device_count()):
    print(f"GPU {gpu_idx} name   :", torch.cuda.get_device_name(gpu_idx))


# 2.0 Run settings

## Purpose
- define the matched downstream SSL settings in one place
- keep the notebook portable across Kaggle and local execution
- make dependencies on earlier notebooks obvious through explicit path variables
- save outputs into clean figures, tables, and JSON folders for GitHub upload

## Process
1. set the active runtime mode and categories.
2. define the feature extraction and evaluation settings used in the final SSL comparison.
3. define all input dependency paths and all output artefact paths before modelling begins.


In [ ]:
#------------------------------------------------------------------------------
# 2.1 Core configuration and output paths
#------------------------------------------------------------------------------
NOTEBOOK_ID = "05_ssl_downstream_models"
RUN_MODE = "full"                  # "debug" or "full"
SEED = 42
DETERMINISTIC_DEBUG = False
set_seed(SEED, deterministic=(RUN_MODE == "debug" and DETERMINISTIC_DEBUG))

DEBUG_CATEGORIES = ["bottle", "carpet", "tile", "transistor"]
CATEGORIES_ALL = [
    "bottle",
    "cable",
    "capsule",
    "carpet",
    "grid",
    "hazelnut",
    "leather",
    "metal_nut",
    "pill",
    "screw",
    "tile",
    "toothbrush",
    "transistor",
    "wood",
    "zipper",
]
CATEGORIES_ACTIVE = DEBUG_CATEGORIES if RUN_MODE == "debug" else CATEGORIES_ALL
QUAL_CATEGORY = next((cat for cat in ["carpet", "tile", "bottle", "transistor"] if cat in CATEGORIES_ACTIVE), CATEGORIES_ACTIVE[0])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_COUNT = torch.cuda.device_count()

IMG_SIZE = 224
VAL_FRAC = 0.10

CPU_COUNT = os.cpu_count() or 2
NUM_WORKERS_BASE = min(4, max(2, CPU_COUNT // 2))
NUM_WORKERS = NUM_WORKERS_BASE if DEVICE == "cuda" else 0
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None

BATCH_SIZE_TRAIN = 32
BATCH_SIZE_TEST = 16

BACKBONE_NAME = "resnet18"
FEATURE_LAYER_NAME = "layer3"
CORESET_RATIO = 0.05
MAX_TRAIN_PATCHES = 120_000
PATCH_SCORE_TOPK = 200
PADIM_DIM = 100
PADIM_EPS = 1e-4

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

if Path("/kaggle/working").exists():
    WORK_ROOT = Path("/kaggle/working/industrial_anomaly_detection_mvtec")
else:
    WORK_ROOT = Path.cwd() / "industrial_anomaly_detection_mvtec"

NOTEBOOK_ROOT = WORK_ROOT / NOTEBOOK_ID / RUN_MODE
FIGURES_DIR = NOTEBOOK_ROOT / "figures"
TABLES_DIR = NOTEBOOK_ROOT / "tables"
JSON_DIR = NOTEBOOK_ROOT / "json"

for folder in [FIGURES_DIR, TABLES_DIR, JSON_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_01_ROOT = WORK_ROOT / "01_dataset_audit_and_splits" / RUN_MODE
NOTEBOOK_03_ROOT = WORK_ROOT / "03_autoencoder_baseline" / RUN_MODE
NOTEBOOK_04_ROOT = WORK_ROOT / "04_simclr_pretraining" / RUN_MODE

SPLIT_MANIFEST_PATH = NOTEBOOK_01_ROOT / "json" / "split_manifest.json"
LEAKAGE_REPORT_PATH = NOTEBOOK_01_ROOT / "json" / "leakage_report.json"

TABLE_BASELINES_ALL_CATEGORY_PATH = NOTEBOOK_03_ROOT / "tables" / "table_baselines_all_category.csv"
TABLE_BASELINES_ALL_MEAN_PATH = NOTEBOOK_03_ROOT / "tables" / "table_baselines_all_mean.csv"
TABLE_BASELINES_ALL_FULL_PATH = NOTEBOOK_03_ROOT / "tables" / "table_baselines_all_full.csv"

TABLE_SIMCLR_SUMMARY_DEP_PATH = NOTEBOOK_04_ROOT / "tables" / "table_simclr_summary.csv"
SIMCLR_RUNS_JSON_DEP_PATH = NOTEBOOK_04_ROOT / "json" / "simclr_runs.json"
SSL_CHECKPOINTS_DIR = NOTEBOOK_04_ROOT / "checkpoints"

RUN_METADATA_PATH = JSON_DIR / "run_metadata.json"
SSL_DOWNSTREAM_SETTINGS_PATH = JSON_DIR / "ssl_downstream_settings.json"
AVAILABLE_SSL_CHECKPOINTS_PATH = JSON_DIR / "available_ssl_checkpoints.json"

TABLE_SSL_DOWNSTREAM_CATEGORY_PATH = TABLES_DIR / "table_ssl_downstream_category.csv"
TABLE_SSL_DOWNSTREAM_MEAN_PATH = TABLES_DIR / "table_ssl_downstream_mean.csv"
TABLE_SSL_DOWNSTREAM_FULL_PATH = TABLES_DIR / "table_ssl_downstream_full.csv"

TABLE_MAIN_COMPARISON_CATEGORY_PATH = TABLES_DIR / "table_main_comparison_category.csv"
TABLE_MAIN_COMPARISON_MEAN_PATH = TABLES_DIR / "table_main_comparison_mean.csv"
TABLE_MAIN_COMPARISON_FULL_PATH = TABLES_DIR / "table_main_comparison_full.csv"
TABLE_EFFICIENCY_SUMMARY_PATH = TABLES_DIR / "table_efficiency_summary.csv"

FIG_MAIN_COMPARISON_PATH = FIGURES_DIR / "fig_main_comparison.png"
FIG_MAIN_CATEGORY_HEATMAP_PATH = FIGURES_DIR / "fig_main_category_heatmap.png"
FIG_SSL_HEATMAPS_PATH = FIGURES_DIR / "fig_ssl_heatmaps.png"

RUN_METADATA = {
    "notebook_id": NOTEBOOK_ID,
    "run_mode": RUN_MODE,
    "seed": SEED,
    "device": DEVICE,
    "gpu_count": GPU_COUNT,
    "timestamp": datetime.now().isoformat(),
}
save_json_overwrite(RUN_METADATA, RUN_METADATA_PATH)

SSL_DOWNSTREAM_SETTINGS = {
    "run_mode": RUN_MODE,
    "seed": SEED,
    "categories_active": CATEGORIES_ACTIVE,
    "image_size": IMG_SIZE,
    "batch_size_train": BATCH_SIZE_TRAIN,
    "batch_size_test": BATCH_SIZE_TEST,
    "backbone_name": BACKBONE_NAME,
    "feature_layer_name": FEATURE_LAYER_NAME,
    "patchcore": {
        "coreset_ratio": CORESET_RATIO,
        "max_train_patches": MAX_TRAIN_PATCHES,
        "score_topk_pixels": PATCH_SCORE_TOPK,
    },
    "padim": {
        "dim": PADIM_DIM,
        "eps": PADIM_EPS,
        "score_topk_pixels": PATCH_SCORE_TOPK,
    },
}
save_json_overwrite(SSL_DOWNSTREAM_SETTINGS, SSL_DOWNSTREAM_SETTINGS_PATH)

print_banner("Run configuration")
print("RUN_MODE         :", RUN_MODE)
print("SEED             :", SEED)
print("DEVICE           :", DEVICE)
print("GPU_COUNT        :", GPU_COUNT)
print("FEATURE_LAYER    :", FEATURE_LAYER_NAME)
print("CORESET_RATIO    :", CORESET_RATIO)
print("PADIM_DIM        :", PADIM_DIM)
print("ACTIVE CATEGORIES:", CATEGORIES_ACTIVE)
print("WORK_ROOT        :", WORK_ROOT)


# 3.0 Dataset and prior artefacts

## Purpose
- resolve the MVTec AD dataset path in a portable way
- load the split manifest and leakage report from notebook 01 instead of rebuilding those objects here
- load the merged baseline tables from notebook 03
- verify that notebook 04 produced usable SimCLR encoder checkpoints before any downstream comparison starts

## Process
1. resolve the dataset location.
2. load the saved split and leakage artefacts and fail clearly if they are missing.
3. load the merged baseline table and the SimCLR training summary.
4. collect the SSL checkpoints that are available for the downstream stage.


In [ ]:
#------------------------------------------------------------------------------
# 3.1 Resolve dataset path and load prior notebook artefacts
#------------------------------------------------------------------------------
DATASET_CANDIDATES = [
    Path(os.environ.get("MVTEC_DIR", "")) if os.environ.get("MVTEC_DIR") else None,
    Path("/kaggle/input/datasets/ipythonx/mvtec-ad"),
    Path("/kaggle/input/mvtec-ad"),
    Path.cwd() / "data" / "raw" / "mvtec-ad",
]

MVTEC_DIR = None
for candidate in DATASET_CANDIDATES:
    if candidate is not None and candidate.exists():
        MVTEC_DIR = candidate
        break

if MVTEC_DIR is None:
    raise FileNotFoundError(
        "MVTec AD dataset path was not found. Set MVTEC_DIR as an environment variable "
        "or attach the dataset in Kaggle before running the notebook."
    )

required_paths = [
    SPLIT_MANIFEST_PATH,
    LEAKAGE_REPORT_PATH,
    TABLE_BASELINES_ALL_FULL_PATH,
    TABLE_SIMCLR_SUMMARY_DEP_PATH,
    SIMCLR_RUNS_JSON_DEP_PATH,
]
missing_required = [str(path) for path in required_paths if not Path(path).exists()]
if missing_required:
    raise FileNotFoundError(
        "One or more dependency artefacts are missing. Run notebooks 01, 03, and 04 first.\n"
        + "\n".join(missing_required)
    )

split_manifest = read_json(SPLIT_MANIFEST_PATH)
leakage_report = read_json(LEAKAGE_REPORT_PATH)

leakage_failures = []
for key, value in leakage_report.items():
    if isinstance(value, (int, float)) and value != 0:
        leakage_failures.append((key, value))
    elif isinstance(value, list) and len(value) > 0:
        leakage_failures.append((key, f"{len(value)} records"))

if leakage_failures:
    raise RuntimeError(
        "Notebook 01 reported non-zero leakage checks. Fix notebook 01 before continuing.\n"
        + "\n".join([f"{k}: {v}" for k, v in leakage_failures])
    )

table_baselines_all_full = pd.read_csv(TABLE_BASELINES_ALL_FULL_PATH)
table_simclr_summary = pd.read_csv(TABLE_SIMCLR_SUMMARY_DEP_PATH)
simclr_runs = read_json(SIMCLR_RUNS_JSON_DEP_PATH)

ssl_ckpt_map = {
    "mild": SSL_CHECKPOINTS_DIR / "simclr_mild_encoder.pt",
    "strong": SSL_CHECKPOINTS_DIR / "simclr_strong_encoder.pt",
}
available_ssl_ckpt_map = {tag: str(path) for tag, path in ssl_ckpt_map.items() if path.exists()}
if len(available_ssl_ckpt_map) == 0:
    raise FileNotFoundError(
        f"No SimCLR encoder checkpoints were found in {SSL_CHECKPOINTS_DIR}. "
        "Run notebook 04 first."
    )

save_json_overwrite(available_ssl_ckpt_map, AVAILABLE_SSL_CHECKPOINTS_PATH)

categories_from_manifest = list(split_manifest.keys())
CATEGORIES_ACTIVE = [cat for cat in CATEGORIES_ACTIVE if cat in categories_from_manifest]
if len(CATEGORIES_ACTIVE) == 0:
    raise RuntimeError("No active categories from the run configuration were found in the split manifest.")

available_categories = sorted([p.name for p in MVTEC_DIR.iterdir() if p.is_dir()])

print_banner("Loaded prior artefacts")
print("Dataset root                :", MVTEC_DIR)
print("Available dataset categories:", available_categories)
print("Split manifest path         :", SPLIT_MANIFEST_PATH)
print("Leakage report path         :", LEAKAGE_REPORT_PATH)
print("Baseline table path         :", TABLE_BASELINES_ALL_FULL_PATH)
print("SimCLR summary path         :", TABLE_SIMCLR_SUMMARY_DEP_PATH)
print("Available SSL checkpoints   :", available_ssl_ckpt_map)
print("Active categories           :", CATEGORIES_ACTIVE)

display(table_simclr_summary)
display(table_baselines_all_full.tail(10))


# 4.0 Shared datasets, transforms, and evaluation helpers

## Purpose
- define the shared dataset wrapper and dataloader helpers for the downstream SSL stage
- keep image transforms and evaluation metrics in one place
- reuse the same image-level and pixel-level metrics as the baseline notebooks so the final comparison stays fair


In [ ]:
#------------------------------------------------------------------------------
# 4.1 Shared transforms, dataset class, dataloaders, and metrics
#------------------------------------------------------------------------------
tfm_imagenet = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


# Load one RGB image.
def load_rgb(path):
    return Image.open(path).convert("RGB")


# Load one binary mask or return an all-zero mask for good images.
def load_mask(path):
    if path is None or (isinstance(path, float) and np.isnan(path)):
        return torch.zeros((1, IMG_SIZE, IMG_SIZE), dtype=torch.float32)

    mask = Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE), resample=Image.NEAREST)
    mask = (np.array(mask, dtype=np.float32) > 0).astype(np.float32)
    return torch.from_numpy(mask)[None, :, :]


# Return items in a consistent format across the shared split manifest.
class MvtecDataset(Dataset):
    def __init__(self, items, mode, img_tfm):
        self.items = items
        self.mode = mode
        self.img_tfm = img_tfm

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        if self.mode in ["train_good", "val_good"]:
            img_path = item
            label = 0
            mask_path = None
        else:
            img_path = item["img_path"]
            label = int(item["label"])
            mask_path = item.get("mask_path", None)

        image = self.img_tfm(load_rgb(img_path))
        mask = load_mask(mask_path)
        return image, int(label), mask, str(img_path)


# Build a DataLoader matched to the active runtime.
def make_loader(items, mode, batch_size, shuffle):
    dataset = MvtecDataset(items=items, mode=mode, img_tfm=tfm_imagenet)

    loader_kwargs = {
        "dataset": dataset,
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": NUM_WORKERS,
        "pin_memory": DEVICE == "cuda",
    }

    if NUM_WORKERS > 0:
        loader_kwargs["persistent_workers"] = PERSISTENT_WORKERS
        loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

    return DataLoader(**loader_kwargs)


# Build the train, validation, and test loaders for one category.
def make_split_loaders(category):
    train_loader = make_loader(split_manifest[category]["train_good"], mode="train_good", batch_size=BATCH_SIZE_TRAIN, shuffle=True)
    val_loader = make_loader(split_manifest[category]["val_good"], mode="val_good", batch_size=BATCH_SIZE_TEST, shuffle=False)
    test_loader = make_loader(split_manifest[category]["test"], mode="test", batch_size=BATCH_SIZE_TEST, shuffle=False)
    return train_loader, val_loader, test_loader


# Compute image-level ROC-AUC safely.
def safe_roc_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, y_score)


# Compute image-level PR-AUC safely.
def safe_pr_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else average_precision_score(y_true, y_score)


# Flatten masks and heatmaps to compute pixel ROC-AUC.
def pixel_roc_auc(masks_list, heatmaps_list):
    if len(masks_list) == 0 or len(heatmaps_list) == 0:
        return np.nan

    y_true = np.concatenate([mask.reshape(-1) for mask in masks_list]).astype(int)
    y_score = np.concatenate([heat.reshape(-1) for heat in heatmaps_list]).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, y_score)


# Rescale a heatmap to 0-1 for plotting.
def norm_01(x):
    x = x.astype(np.float32)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)


# Convert a normalised tensor image back to display space.
def tensor_to_display(img_tensor):
    image = img_tensor.detach().cpu().permute(1, 2, 0).numpy()
    image = image * np.array(IMAGENET_STD)[None, None, :] + np.array(IMAGENET_MEAN)[None, None, :]
    return np.clip(image, 0.0, 1.0)


# Draw an image with an optional heatmap overlay.
def overlay(ax, img_tensor, heat_2d=None, title=""):
    ax.imshow(tensor_to_display(img_tensor))
    if heat_2d is not None:
        ax.imshow(norm_01(heat_2d), alpha=0.45)
    ax.set_title(title, fontsize=10)
    ax.axis("off")


# Draw a grayscale mask.
def show_mask(ax, mask_2d, title=""):
    ax.imshow(mask_2d, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.axis("off")


# Evaluate one scoring function across the test set.
def eval_method(test_loader, score_fn):
    y_true = []
    y_score = []
    masks = []
    heats = []

    eval_start = time.time()
    eval_image_n = 0

    for images, labels, batch_masks, _ in test_loader:
        scores, heatmaps = score_fn(images)

        y_true.extend(labels.numpy().tolist())
        y_score.extend(scores.tolist())

        mask_np = batch_masks.squeeze(1).numpy()
        for idx in range(mask_np.shape[0]):
            masks.append(mask_np[idx])
            heats.append(heatmaps[idx])

        eval_image_n += images.shape[0]

    total_eval_sec = time.time() - eval_start

    return {
        "img_roc_auc": safe_roc_auc(y_true, y_score),
        "img_pr_auc": safe_pr_auc(y_true, y_score),
        "px_roc_auc": pixel_roc_auc(masks, heats),
        "sec_per_img": total_eval_sec / max(eval_image_n, 1),
        "n_eval_images": int(eval_image_n),
        "y_true": y_true,
        "y_score": y_score,
        "masks": masks,
        "heats": heats,
    }


# Keep a small mixed good / anomaly subset for the qualitative figure.
def select_qual_items(test_items, n_good=3, n_bad=3):
    good_items = [item for item in test_items if int(item["label"]) == 0][:n_good]
    bad_items = [item for item in test_items if int(item["label"]) == 1][:n_bad]
    return good_items + bad_items


# 5.0 SSL downstream model helpers

## Purpose
- rebuild the SimCLR encoder checkpoints saved in notebook 04
- define the shared feature extraction, PatchCore, and PaDiM helpers needed for the downstream SSL stage
- keep the main comparison loop shorter by moving most method logic into reusable runtime builders


In [ ]:
#------------------------------------------------------------------------------
# 5.1 Backbone, feature hooks, PatchCore, PaDiM, and runtime builders
#------------------------------------------------------------------------------
def get_resnet18_ssl():
    model = torchvision.models.resnet18(weights=None)
    model.fc = nn.Identity()
    model.eval()
    return model


# Register hooks so intermediate feature maps can be collected.
def make_feature_hooks(model, layer_names):
    features = {}
    handles = []

    def hook_factory(layer_name):
        def _hook(_, __, output):
            features[layer_name] = output
        return _hook

    for layer_name in layer_names:
        handle = getattr(model, layer_name).register_forward_hook(hook_factory(layer_name))
        handles.append(handle)

    return features, handles


# Remove hook handles after the runtime is no longer needed.
def remove_handles(handles):
    for handle in handles:
        handle.remove()


@torch.inference_mode()
def forward_get_feats(model, features, inputs, layer_names):
    _ = model(inputs)
    return [features[layer_name] for layer_name in layer_names]


# Flatten a feature map into patch rows.
def fmap_to_patches(feature_map):
    batch_n, channels, height, width = feature_map.shape
    return feature_map.permute(0, 2, 3, 1).reshape(batch_n, height * width, channels)


# Upsample and concatenate patch features from one or more layers.
def concat_patch_features(feature_maps):
    target_h, target_w = feature_maps[-1].shape[-2:]
    patch_list = []

    for feature_map in feature_maps:
        if feature_map.shape[-2:] != (target_h, target_w):
            feature_map = F.interpolate(
                feature_map,
                size=(target_h, target_w),
                mode="bilinear",
                align_corners=False,
            )
        patch_list.append(fmap_to_patches(feature_map))

    return torch.cat(patch_list, dim=-1)


# Build a FAISS L2 index for nearest-neighbour search.
def faiss_index_l2(array_2d):
    array_2d = np.asarray(array_2d, dtype=np.float32)
    index = faiss.IndexFlatL2(array_2d.shape[1])
    index.add(array_2d)
    return index


# Rebuild one saved SimCLR encoder checkpoint.
def load_ssl_encoder(ckpt_path):
    model = get_resnet18_ssl()
    state = torch.load(ckpt_path, map_location="cpu")

    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]

    if isinstance(state, dict):
        clean_state = {}
        for key, value in state.items():
            clean_key = key.replace("module.", "", 1) if key.startswith("module.") else key
            clean_state[clean_key] = value
        state = clean_state

    model.load_state_dict(state, strict=True)
    model.eval()
    return model


# Collect normal train patch embeddings and keep a deterministic coreset subset.
def build_patch_bank(model, features, train_loader, layer_names, category, aug_strength):
    patch_chunks = []
    total_patches = 0

    for images, _, _, _ in train_loader:
        images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
        feature_maps = forward_get_feats(model, features, images, layer_names)
        patches = concat_patch_features(feature_maps).detach().cpu().numpy()
        patches = patches.reshape(-1, patches.shape[-1]).astype(np.float32)
        patch_chunks.append(patches)
        total_patches += patches.shape[0]

        if total_patches >= MAX_TRAIN_PATCHES:
            break

    bank_full = np.concatenate(patch_chunks, axis=0).astype(np.float32)

    if len(bank_full) > MAX_TRAIN_PATCHES:
        rng_cap = np.random.default_rng(stable_seed_from_text(SEED, f"{category}_{aug_strength}_patch_bank_cap"))
        idx_cap = rng_cap.choice(len(bank_full), size=MAX_TRAIN_PATCHES, replace=False)
        bank_full = bank_full[idx_cap]

    keep_n = max(1, int(round(len(bank_full) * float(CORESET_RATIO))))
    keep_n = min(keep_n, len(bank_full))

    rng_core = np.random.default_rng(stable_seed_from_text(SEED, f"{category}_{aug_strength}_patchcore_coreset"))
    coreset_idx = rng_core.choice(len(bank_full), size=keep_n, replace=False)
    bank = bank_full[coreset_idx]

    return bank


@torch.inference_mode()
def patchcore_scores(model, features, images, layer_names, index):
    images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
    feature_maps = forward_get_feats(model, features, images, layer_names)
    patches = concat_patch_features(feature_maps).detach().cpu().numpy()

    batch_n, patch_n, feat_dim = patches.shape
    queries = patches.reshape(-1, feat_dim).astype(np.float32)
    dist2, _ = index.search(queries, 1)
    dist2 = dist2.reshape(batch_n, patch_n)

    feat_h, feat_w = feature_maps[-1].shape[-2:]
    heat = dist2.reshape(batch_n, feat_h, feat_w)
    heat_tensor = torch.from_numpy(heat).unsqueeze(1)
    heat_up = F.interpolate(
        heat_tensor,
        size=(IMG_SIZE, IMG_SIZE),
        mode="bilinear",
        align_corners=False,
    ).squeeze(1).numpy()

    flat = heat_up.reshape(batch_n, -1)
    topk_n = min(PATCH_SCORE_TOPK, flat.shape[1])
    scores = np.mean(np.sort(flat, axis=1)[:, -topk_n:], axis=1)

    return scores, heat_up


# Fit PaDiM Gaussian statistics from normal train patches.
def padim_fit(model, features, train_loader, layer_name, category, aug_strength):
    all_feature_maps = []

    for images, _, _, _ in train_loader:
        images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
        feature_map = forward_get_feats(model, features, images, [layer_name])[0].detach().cpu()
        all_feature_maps.append(feature_map)

    feature_map = torch.cat(all_feature_maps, dim=0)
    sample_n, channels, feat_h, feat_w = feature_map.shape

    dim = min(PADIM_DIM, channels)
    rng_dim = np.random.default_rng(stable_seed_from_text(SEED, f"{category}_{aug_strength}_padim_dim"))
    dim_idx = rng_dim.choice(channels, size=dim, replace=False)

    feature_map = feature_map[:, dim_idx, :, :]
    feats_np = feature_map.permute(0, 2, 3, 1).reshape(sample_n, feat_h * feat_w, dim).numpy()

    mu = feats_np.mean(axis=0)
    cov_inv = []

    eye = np.eye(dim, dtype=np.float32)
    for patch_idx in range(feat_h * feat_w):
        patch_features = feats_np[:, patch_idx, :]
        cov = np.cov(patch_features, rowvar=False).astype(np.float32) + PADIM_EPS * eye
        cov_inv.append(np.linalg.inv(cov).astype(np.float32))

    cov_inv = np.stack(cov_inv, axis=0)

    return {
        "dim_idx": dim_idx.astype(np.int64),
        "mu": mu.astype(np.float32),
        "cov_inv": cov_inv.astype(np.float32),
        "H": int(feat_h),
        "W": int(feat_w),
        "D": int(dim),
    }


@torch.inference_mode()
def padim_scores(model, features, images, layer_name, stats):
    images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
    feature_map = forward_get_feats(model, features, images, [layer_name])[0].detach().cpu()

    dim_idx = torch.tensor(stats["dim_idx"])
    feature_map = feature_map[:, dim_idx, :, :]

    batch_n, feat_dim, feat_h, feat_w = feature_map.shape
    feats_np = feature_map.permute(0, 2, 3, 1).reshape(batch_n, feat_h * feat_w, feat_dim).numpy()

    mu = stats["mu"]
    cov_inv = stats["cov_inv"]

    heat = np.zeros((batch_n, feat_h * feat_w), dtype=np.float32)
    for patch_idx in range(feat_h * feat_w):
        diff = feats_np[:, patch_idx, :] - mu[patch_idx]
        tmp = diff @ cov_inv[patch_idx]
        heat[:, patch_idx] = np.sum(tmp * diff, axis=1)

    heat = heat.reshape(batch_n, feat_h, feat_w)
    heat_tensor = torch.from_numpy(heat).unsqueeze(1)
    heat_up = F.interpolate(
        heat_tensor,
        size=(IMG_SIZE, IMG_SIZE),
        mode="bilinear",
        align_corners=False,
    ).squeeze(1).numpy()

    flat = heat_up.reshape(batch_n, -1)
    topk_n = min(PATCH_SCORE_TOPK, flat.shape[1])
    scores = np.mean(np.sort(flat, axis=1)[:, -topk_n:], axis=1)

    return scores, heat_up


# Build the runtime for one SSL downstream method.
def build_ssl_runtime(category, aug_strength, anomaly_head):
    train_loader, val_loader, test_loader = make_split_loaders(category)

    ckpt_path = Path(available_ssl_ckpt_map[aug_strength])
    model = load_ssl_encoder(ckpt_path).to(DEVICE)
    features, handles = make_feature_hooks(model, [FEATURE_LAYER_NAME])

    reset_peak_gpu_memory()
    fit_start = time.time()

    if anomaly_head == "patchcore":
        bank = build_patch_bank(
            model=model,
            features=features,
            train_loader=train_loader,
            layer_names=[FEATURE_LAYER_NAME],
            category=category,
            aug_strength=aug_strength,
        )
        index = faiss_index_l2(bank)

        def score_fn(images):
            return patchcore_scores(model, features, images, [FEATURE_LAYER_NAME], index)

        fit_stats = {
            "feature_dim": int(bank.shape[1]),
            "memory_bank_n": int(bank.shape[0]),
            "memory_bank_mb": float(bank.nbytes / (1024 ** 2)),
        }

    elif anomaly_head == "padim":
        stats = padim_fit(
            model=model,
            features=features,
            train_loader=train_loader,
            layer_name=FEATURE_LAYER_NAME,
            category=category,
            aug_strength=aug_strength,
        )

        def score_fn(images):
            return padim_scores(model, features, images, FEATURE_LAYER_NAME, stats)

        fit_stats = {
            "feature_dim": int(stats["D"]),
            "memory_bank_n": int(stats["H"] * stats["W"]),
            "memory_bank_mb": float((stats["mu"].nbytes + stats["cov_inv"].nbytes) / (1024 ** 2)),
        }

    else:
        remove_handles(handles)
        raise ValueError(f"Unsupported anomaly head: {anomaly_head}")

    fit_sec = float(time.time() - fit_start)
    peak_gpu_mem_mb = get_peak_gpu_memory_mb()

    return {
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "model": model,
        "features": features,
        "handles": handles,
        "score_fn": score_fn,
        "feature_dim": fit_stats["feature_dim"],
        "memory_bank_n": fit_stats["memory_bank_n"],
        "memory_bank_mb": fit_stats["memory_bank_mb"],
        "fit_sec": fit_sec,
        "peak_gpu_mem_mb": peak_gpu_mem_mb,
        "checkpoint_size_mb": float(file_size_mb(ckpt_path)),
    }


# Keep one detailed subset for the qualitative SSL heatmap figure.
def collect_qualitative_cache(category, score_fn):
    qual_items = select_qual_items(split_manifest[category]["test"], n_good=3, n_bad=3)
    if len(qual_items) == 0:
        return None

    qual_loader = make_loader(qual_items, mode="test", batch_size=len(qual_items), shuffle=False)
    images, labels, masks, paths = next(iter(qual_loader))
    scores, heats = score_fn(images)

    return {
        "images": images,
        "labels": labels.numpy(),
        "masks": masks.squeeze(1).numpy(),
        "paths": paths,
        "scores": scores,
        "heats": heats,
    }


# 6.0 Run the downstream SSL comparison

## Purpose
- evaluate the saved SimCLR encoders with PatchCore and PaDiM under matched settings
- save SSL-only tables first so the downstream stage is reviewable on its own
- merge the SSL results with the baseline rows from notebook 03 to create the final comparison table used later in the report


In [ ]:
#------------------------------------------------------------------------------
# 6.1 Run the downstream SSL methods and save tables
#------------------------------------------------------------------------------
method_specs = []
for aug_strength in ["mild", "strong"]:
    if aug_strength in available_ssl_ckpt_map:
        method_specs.append({"method": f"simclr_{aug_strength}_patchcore", "aug_strength": aug_strength, "anomaly_head": "patchcore"})
        method_specs.append({"method": f"simclr_{aug_strength}_padim", "aug_strength": aug_strength, "anomaly_head": "padim"})

if len(method_specs) == 0:
    raise RuntimeError("No SSL downstream methods were constructed because no encoder checkpoints were available.")

ordered_cols = [
    "category",
    "method",
    "run_mode",
    "representation_source",
    "anomaly_head",
    "aug_strength",
    "backbone_name",
    "layers",
    "img_size",
    "coreset_ratio",
    "n_train_good",
    "n_val_good",
    "n_test_total",
    "n_test_anomaly",
    "img_roc_auc",
    "img_pr_auc",
    "px_roc_auc",
    "fit_sec",
    "sec_per_img",
    "n_eval_images",
    "feature_dim",
    "memory_bank_n",
    "memory_bank_mb",
    "checkpoint_size_mb",
    "peak_gpu_mem_mb",
]

rows_ssl = []
qual_cache_ssl = None

print_banner("Running SSL downstream methods")
print("Methods:", [spec["method"] for spec in method_specs])

for category in CATEGORIES_ACTIVE:
    print("\n" + "-" * 90)
    print("CATEGORY:", category)
    print("-" * 90)

    n_train_good = len(split_manifest[category]["train_good"])
    n_val_good = len(split_manifest[category]["val_good"])
    n_test_total = len(split_manifest[category]["test"])
    n_test_anomaly = int(sum(int(item["label"]) == 1 for item in split_manifest[category]["test"]))

    for spec in method_specs:
        runtime = build_ssl_runtime(
            category=category,
            aug_strength=spec["aug_strength"],
            anomaly_head=spec["anomaly_head"],
        )
        results = eval_method(runtime["test_loader"], runtime["score_fn"])

        rows_ssl.append(
            {
                "category": category,
                "method": spec["method"],
                "run_mode": RUN_MODE,
                "representation_source": "simclr",
                "anomaly_head": spec["anomaly_head"],
                "aug_strength": spec["aug_strength"],
                "backbone_name": BACKBONE_NAME,
                "layers": FEATURE_LAYER_NAME,
                "img_size": IMG_SIZE,
                "coreset_ratio": CORESET_RATIO if spec["anomaly_head"] == "patchcore" else np.nan,
                "n_train_good": n_train_good,
                "n_val_good": n_val_good,
                "n_test_total": n_test_total,
                "n_test_anomaly": n_test_anomaly,
                "img_roc_auc": results["img_roc_auc"],
                "img_pr_auc": results["img_pr_auc"],
                "px_roc_auc": results["px_roc_auc"],
                "fit_sec": runtime["fit_sec"],
                "sec_per_img": results["sec_per_img"],
                "n_eval_images": results["n_eval_images"],
                "feature_dim": runtime["feature_dim"],
                "memory_bank_n": runtime["memory_bank_n"],
                "memory_bank_mb": runtime["memory_bank_mb"],
                "checkpoint_size_mb": runtime["checkpoint_size_mb"],
                "peak_gpu_mem_mb": runtime["peak_gpu_mem_mb"],
            }
        )

        if category == QUAL_CATEGORY and qual_cache_ssl is None and spec["method"] == "simclr_mild_padim":
            qual_cache_ssl = collect_qualitative_cache(category, runtime["score_fn"])
            if qual_cache_ssl is not None:
                qual_cache_ssl["method"] = spec["method"]
                qual_cache_ssl["category"] = category

        print(
            spec["method"],
            {
                "img_roc_auc": results["img_roc_auc"],
                "img_pr_auc": results["img_pr_auc"],
                "px_roc_auc": results["px_roc_auc"],
                "fit_sec": runtime["fit_sec"],
                "sec_per_img": results["sec_per_img"],
            },
        )

        remove_handles(runtime["handles"])
        del runtime["model"]
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

df_ssl_downstream_category = pd.DataFrame(rows_ssl)[ordered_cols]

mean_metric_cols = [
    "n_train_good",
    "n_val_good",
    "n_test_total",
    "n_test_anomaly",
    "img_roc_auc",
    "img_pr_auc",
    "px_roc_auc",
    "fit_sec",
    "sec_per_img",
    "n_eval_images",
    "feature_dim",
    "memory_bank_n",
    "memory_bank_mb",
    "checkpoint_size_mb",
    "peak_gpu_mem_mb",
]

df_ssl_downstream_mean = (
    df_ssl_downstream_category
    .groupby(["method", "representation_source", "anomaly_head", "aug_strength"], as_index=False)[mean_metric_cols]
    .mean(numeric_only=True)
)
df_ssl_downstream_mean["category"] = "MEAN"
df_ssl_downstream_mean["run_mode"] = RUN_MODE
df_ssl_downstream_mean["backbone_name"] = BACKBONE_NAME
df_ssl_downstream_mean["layers"] = FEATURE_LAYER_NAME
df_ssl_downstream_mean["img_size"] = IMG_SIZE
df_ssl_downstream_mean["coreset_ratio"] = df_ssl_downstream_mean["anomaly_head"].map({"patchcore": CORESET_RATIO, "padim": np.nan})
df_ssl_downstream_mean = df_ssl_downstream_mean[ordered_cols]

df_ssl_downstream_full = pd.concat(
    [df_ssl_downstream_category, df_ssl_downstream_mean],
    ignore_index=True,
)

save_csv_overwrite(df_ssl_downstream_category, TABLE_SSL_DOWNSTREAM_CATEGORY_PATH)
save_csv_overwrite(df_ssl_downstream_mean, TABLE_SSL_DOWNSTREAM_MEAN_PATH)
save_csv_overwrite(df_ssl_downstream_full, TABLE_SSL_DOWNSTREAM_FULL_PATH)

baseline_keep = ["imagenet_patchcore", "imagenet_padim", "autoencoder"]
df_baselines_core = table_baselines_all_full.copy()
df_baselines_core = df_baselines_core[df_baselines_core["category"] != "MEAN"].copy()
df_baselines_core = df_baselines_core[df_baselines_core["method"].isin(baseline_keep)].copy()
df_baselines_core = df_baselines_core[df_baselines_core["category"].isin(CATEGORIES_ACTIVE)].copy()
df_baselines_core = df_baselines_core[ordered_cols]

df_main_comparison_category = pd.concat(
    [df_baselines_core, df_ssl_downstream_category],
    ignore_index=True,
)

df_main_comparison_mean = (
    df_main_comparison_category
    .groupby(["method", "representation_source", "anomaly_head", "aug_strength"], as_index=False)[mean_metric_cols]
    .mean(numeric_only=True)
)
df_main_comparison_mean["category"] = "MEAN"
df_main_comparison_mean["run_mode"] = RUN_MODE
df_main_comparison_mean["backbone_name"] = ""
df_main_comparison_mean["layers"] = FEATURE_LAYER_NAME
df_main_comparison_mean["img_size"] = IMG_SIZE
df_main_comparison_mean["coreset_ratio"] = df_main_comparison_mean["anomaly_head"].map(
    {"patchcore": CORESET_RATIO, "padim": np.nan, "autoencoder": np.nan}
)
df_main_comparison_mean = df_main_comparison_mean[ordered_cols]

df_main_comparison_full = pd.concat(
    [df_main_comparison_category, df_main_comparison_mean],
    ignore_index=True,
)

table_efficiency_summary = (
    df_main_comparison_mean[
        [
            "method",
            "representation_source",
            "anomaly_head",
            "aug_strength",
            "img_roc_auc",
            "img_pr_auc",
            "px_roc_auc",
            "fit_sec",
            "sec_per_img",
            "feature_dim",
            "memory_bank_n",
            "memory_bank_mb",
            "checkpoint_size_mb",
            "peak_gpu_mem_mb",
        ]
    ]
    .sort_values(["img_pr_auc", "img_roc_auc"], ascending=False)
    .reset_index(drop=True)
)

save_csv_overwrite(df_main_comparison_category, TABLE_MAIN_COMPARISON_CATEGORY_PATH)
save_csv_overwrite(df_main_comparison_mean, TABLE_MAIN_COMPARISON_MEAN_PATH)
save_csv_overwrite(df_main_comparison_full, TABLE_MAIN_COMPARISON_FULL_PATH)
save_csv_overwrite(table_efficiency_summary, TABLE_EFFICIENCY_SUMMARY_PATH)

print_banner("SSL downstream mean table")
display(df_ssl_downstream_mean.sort_values("img_pr_auc", ascending=False))

print_banner("Main comparison mean table")
display(df_main_comparison_mean.sort_values("img_pr_auc", ascending=False))


# 7.0 Save comparison figures

## Purpose
- export one compact multi-metric summary figure for the main comparison
- save one category-level heatmap focused on image PR-AUC because that metric is the clearest headline metric in the report
- save one small qualitative SSL heatmap figure so the repo has an immediately viewable visual output outside the notebooks


In [ ]:
#------------------------------------------------------------------------------
# 7.1 Main comparison figures
#------------------------------------------------------------------------------
METHOD_ORDER = [
    "imagenet_patchcore",
    "imagenet_padim",
    "autoencoder",
    "simclr_mild_patchcore",
    "simclr_strong_patchcore",
    "simclr_mild_padim",
    "simclr_strong_padim",
]

plot_df = df_main_comparison_mean.copy()
plot_df["method"] = pd.Categorical(plot_df["method"], categories=METHOD_ORDER, ordered=True)
plot_df = plot_df.sort_values("method")

fig = plt.figure(figsize=(13, 8))

ax1 = plt.subplot(2, 2, 1)
use_df = plot_df.dropna(subset=["img_roc_auc"])
ax1.bar(use_df["method"].astype(str), use_df["img_roc_auc"])
ax1.set_title("Mean image ROC-AUC")
ax1.set_ylim(0, 1.02)
ax1.tick_params(axis="x", rotation=25)

ax2 = plt.subplot(2, 2, 2)
use_df = plot_df.dropna(subset=["img_pr_auc"])
ax2.bar(use_df["method"].astype(str), use_df["img_pr_auc"])
ax2.set_title("Mean image PR-AUC")
ax2.set_ylim(0, 1.02)
ax2.tick_params(axis="x", rotation=25)

ax3 = plt.subplot(2, 2, 3)
use_df = plot_df.dropna(subset=["px_roc_auc"])
ax3.bar(use_df["method"].astype(str), use_df["px_roc_auc"])
ax3.set_title("Mean pixel ROC-AUC")
ax3.set_ylim(0, 1.02)
ax3.tick_params(axis="x", rotation=25)

ax4 = plt.subplot(2, 2, 4)
use_df = plot_df.dropna(subset=["sec_per_img"])
ax4.bar(use_df["method"].astype(str), use_df["sec_per_img"])
ax4.set_title("Mean inference sec / image")
ax4.tick_params(axis="x", rotation=25)

plt.tight_layout()
ensure_parent(FIG_MAIN_COMPARISON_PATH)
plt.savefig(FIG_MAIN_COMPARISON_PATH, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved figure:", FIG_MAIN_COMPARISON_PATH)

heat_df = (
    df_main_comparison_category
    .pivot(index="category", columns="method", values="img_pr_auc")
    .reindex(index=CATEGORIES_ACTIVE, columns=METHOD_ORDER)
)

fig = plt.figure(figsize=(12, 6))
plt.imshow(heat_df.values, aspect="auto")
plt.xticks(range(len(heat_df.columns)), heat_df.columns, rotation=25, ha="right")
plt.yticks(range(len(heat_df.index)), heat_df.index)
plt.title("Category image PR-AUC heatmap")
plt.colorbar()
plt.tight_layout()
ensure_parent(FIG_MAIN_CATEGORY_HEATMAP_PATH)
plt.savefig(FIG_MAIN_CATEGORY_HEATMAP_PATH, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved figure:", FIG_MAIN_CATEGORY_HEATMAP_PATH)

if qual_cache_ssl is not None:
    qual_images = qual_cache_ssl["images"]
    qual_labels = qual_cache_ssl["labels"]
    qual_masks = qual_cache_ssl["masks"]
    qual_paths = qual_cache_ssl["paths"]
    qual_scores = qual_cache_ssl["scores"]
    qual_heats = qual_cache_ssl["heats"]

    row_n = qual_images.shape[0]
    plt.figure(figsize=(12, 3.0 * row_n))

    for row_idx in range(row_n):
        row_name = f"{'anomaly' if qual_labels[row_idx] == 1 else 'good'} | {Path(qual_paths[row_idx]).parent.name}"

        ax = plt.subplot(row_n, 3, row_idx * 3 + 1)
        overlay(ax, qual_images[row_idx], None, title="Image")
        ax.set_ylabel(row_name, fontsize=9)

        ax = plt.subplot(row_n, 3, row_idx * 3 + 2)
        show_mask(ax, qual_masks[row_idx], title="GT mask")

        ax = plt.subplot(row_n, 3, row_idx * 3 + 3)
        overlay(ax, qual_images[row_idx], qual_heats[row_idx], title=f"{qual_cache_ssl['method']}\nscore={qual_scores[row_idx]:.3f}")

    plt.tight_layout()
    ensure_parent(FIG_SSL_HEATMAPS_PATH)
    plt.savefig(FIG_SSL_HEATMAPS_PATH, dpi=220, bbox_inches="tight")
    plt.show()
    print("Saved figure:", FIG_SSL_HEATMAPS_PATH)


# 8.0 Final review

## Purpose
- print the saved artefact paths in one place
- make it easy to confirm that the notebook finished and produced the tables needed by later notebooks


In [ ]:
#------------------------------------------------------------------------------
# 8.1 Final saved artefacts
#------------------------------------------------------------------------------
print_banner("Saved artefacts")
print("Run metadata path                 :", RUN_METADATA_PATH)
print("Downstream settings path          :", SSL_DOWNSTREAM_SETTINGS_PATH)
print("Available checkpoints path        :", AVAILABLE_SSL_CHECKPOINTS_PATH)
print("SSL category table path           :", TABLE_SSL_DOWNSTREAM_CATEGORY_PATH)
print("SSL mean table path               :", TABLE_SSL_DOWNSTREAM_MEAN_PATH)
print("SSL full table path               :", TABLE_SSL_DOWNSTREAM_FULL_PATH)
print("Main comparison category path     :", TABLE_MAIN_COMPARISON_CATEGORY_PATH)
print("Main comparison mean path         :", TABLE_MAIN_COMPARISON_MEAN_PATH)
print("Main comparison full path         :", TABLE_MAIN_COMPARISON_FULL_PATH)
print("Efficiency summary path           :", TABLE_EFFICIENCY_SUMMARY_PATH)
print("Main comparison figure path       :", FIG_MAIN_COMPARISON_PATH)
print("Main category heatmap figure path :", FIG_MAIN_CATEGORY_HEATMAP_PATH)
print("SSL qualitative figure path       :", FIG_SSL_HEATMAPS_PATH)

display(df_main_comparison_mean.sort_values("img_pr_auc", ascending=False))
